# TP 5 — Random Forest
**Machine Learning · Apprentissage supervisé · Ensemble d’arbres (bagging)**

Ce TP reprend les exemples de la présentation (diapos 41 à 45) :

| Exemple | Diapos | Problème |
|---|---|---|
| Exemple 1 | 42 – 43 | Classification : va-t-on jouer au tennis demain ? 3 arbres, bootstrap, 2 variables tirées |
| Exemple 2 | 44 | Régression : ventes de glaces à 28 °C, 3 arbres, moyenne |

**Objectifs** : programmer **en Python pur** le bootstrap, l’arbre à une question (`max_depth = 1`) avec un choix limité de variables (`max_features = 2`), le vote et la moyenne ; appliquer ces fonctions aux exemples des diapos ; puis refaire les mêmes exemples avec **scikit-learn** et comparer les deux implémentations.

## Rappel de cours

Une forêt aléatoire = **plusieurs arbres de décision** + **deux sources de hasard** :
1. **Bootstrap** : chaque arbre est appris sur un échantillon de $n$ exemples **tirés avec remise** (≈ 63 % d’exemples distincts). Les exemples jamais tirés sont **out-of-bag** (OOB).
2. **Variables tirées** : à chaque question, l’arbre ne peut choisir qu’entre $m$ variables tirées au hasard (`max_features`).

**Agrégation** : classification → **vote** majoritaire ; régression → **moyenne** $\hat y = \frac{1}{B}\sum_b \text{arbre}_b(x)$.

Chaque arbre est construit exactement comme au TP 4 (entropie et gain, ou SSE).

In [1]:
import math, random

---
# Partie A — Implémentation from scratch (Python pur)

## Exemple 1 : tennis, 3 arbres

Mêmes 8 jours que le TP 4. Demain : **Soleil, humidité 72 %, vent fort**.

In [2]:
jours = {
    "J1": {"ciel": "Soleil", "humidite": 65, "vent": "Faible", "joue": "Oui"},
    "J2": {"ciel": "Soleil", "humidite": 90, "vent": "Faible", "joue": "Non"},
    "J3": {"ciel": "Pluie",  "humidite": 70, "vent": "Faible", "joue": "Oui"},
    "J4": {"ciel": "Soleil", "humidite": 85, "vent": "Faible", "joue": "Non"},
    "J5": {"ciel": "Pluie",  "humidite": 80, "vent": "Fort",   "joue": "Non"},
    "J6": {"ciel": "Soleil", "humidite": 70, "vent": "Fort",   "joue": "Non"},
    "J7": {"ciel": "Pluie",  "humidite": 60, "vent": "Faible", "joue": "Oui"},
    "J8": {"ciel": "Soleil", "humidite": 75, "vent": "Faible", "joue": "Oui"},
}
demain = {"ciel": "Soleil", "humidite": 72, "vent": "Fort"}

# Les 3 échantillons bootstrap et les variables tirées, donnés dans les diapos 42 – 43
echantillons = {
    "Arbre 1": (["J1", "J3", "J3", "J5", "J6", "J6", "J8", "J2"], ["ciel", "vent"]),
    "Arbre 2": (["J2", "J2", "J4", "J5", "J7", "J7", "J8", "J1"], ["ciel", "humidite"]),
    "Arbre 3": (["J1", "J4", "J4", "J5", "J5", "J6", "J8", "J3"], ["ciel", "vent"]),
}

### Question 1 — Le bootstrap : tirage avec remise et exemples out-of-bag
**Réponse.** Dans un échantillon bootstrap, certains jours apparaissent plusieurs fois et d’autres jamais : ce sont les jours **out-of-bag**, utilisables pour tester l’arbre. La diapo 42 annonce : arbre 1 → J4, J7 ; arbre 2 → J3, J6 ; arbre 3 → J2, J7.

En moyenne, un tirage de $n$ exemples avec remise contient $1 - (1 - 1/n)^n \approx 63\,\%$ d’exemples distincts. On le vérifie par simulation.

In [3]:
for nom, (ech, _) in echantillons.items():
    oob = sorted(set(jours) - set(ech))
    print(f"{nom} : distincts = {len(set(ech))}/8 ; out-of-bag = {oob}")

def bootstrap(exemples, rng):
    # Tirer len(exemples) éléments AVEC remise
    return [rng.choice(exemples) for _ in exemples]

rng = random.Random(0)
n = 1000
fractions = [len(set(bootstrap(list(range(n)), rng))) / n for _ in range(200)]
print(f"Proportion moyenne d'exemples distincts : {sum(fractions) / len(fractions):.3f}  (théorie : 1 − 1/e = {1 - math.exp(-1):.3f})")

Arbre 1 : distincts = 6/8 ; out-of-bag = ['J4', 'J7']
Arbre 2 : distincts = 6/8 ; out-of-bag = ['J3', 'J6']
Arbre 3 : distincts = 6/8 ; out-of-bag = ['J2', 'J7']
Proportion moyenne d'exemples distincts : 0.633  (théorie : 1 − 1/e = 0.632)


### Question 2 — Apprendre chaque arbre sur son échantillon (une question, 2 variables autorisées)
**Réponse.** Même calcul que le TP 4, mais sur l’échantillon bootstrap (avec ses doublons) et seulement avec les variables tirées.
- Arbre 1 : 4 O / 4 N → $H = 1$ ; *Vent fort ?* → gain $= 1 - \frac{5}{8}\times 0{,}722 = 0{,}55$
- Arbre 2 : *Humidité ≤ 77,5 ?* donne 2 groupes purs → gain $= 1$ (l’arbre 2 n’a pas le droit de regarder le vent)
- Arbre 3 : 3 O / 5 N → $H = 0{,}954$ ; *Vent fort ?* → gain $= 0{,}954 - \frac{5}{8}\times 0{,}971 = 0{,}35$

In [4]:
def entropie(etiquettes):
    n = len(etiquettes)
    return -sum((etiquettes.count(c) / n) * math.log2(etiquettes.count(c) / n) for c in set(etiquettes)) if n else 0.0

def questions(donnees, variables):
    qs = []
    if "ciel" in variables:
        qs.append(("ciel = Soleil ?", lambda j: j["ciel"] == "Soleil"))
    if "vent" in variables:
        qs.append(("vent fort ?", lambda j: j["vent"] == "Fort"))
    if "humidite" in variables:
        v = sorted(set(j["humidite"] for j in donnees))
        for a, b in zip(v, v[1:]):
            t = (a + b) / 2
            qs.append((f"humidité ≤ {t} ?", lambda j, t=t: j["humidite"] <= t))
    return qs

def majoritaire(etiquettes):
    return max(set(etiquettes), key=etiquettes.count)

def apprendre_souche(donnees, variables):
    # Arbre à UNE question (max_depth = 1) : on garde la question au plus grand gain
    y = [j["joue"] for j in donnees]
    meilleur = None
    for nom, test in questions(donnees, variables):
        oui = [j["joue"] for j in donnees if test(j)]
        non = [j["joue"] for j in donnees if not test(j)]
        if not oui or not non:
            continue
        g = entropie(y) - len(oui) / len(y) * entropie(oui) - len(non) / len(y) * entropie(non)
        if meilleur is None or g > meilleur["gain"]:
            meilleur = {"question": nom, "test": test, "gain": g, "H": entropie(y),
                        "feuille_oui": majoritaire(oui), "feuille_non": majoritaire(non),
                        "oui": oui, "non": non}
    return meilleur

foret = {}
for nom, (ech, variables) in echantillons.items():
    donnees = [jours[j] for j in ech]                 # les doublons comptent deux fois
    a = apprendre_souche(donnees, variables)
    foret[nom] = a
    print(f"{nom} : H = {a['H']:.3f} ; question = {a['question']} (gain {a['gain']:.3f}) ; "
          f"oui → {a['feuille_oui']} ({a['oui'].count('Oui')} O / {a['oui'].count('Non')} N) ; "
          f"non → {a['feuille_non']} ({a['non'].count('Oui')} O / {a['non'].count('Non')} N)")

Arbre 1 : H = 1.000 ; question = vent fort ? (gain 0.549) ; oui → Non (0 O / 3 N) ; non → Oui (4 O / 1 N)
Arbre 2 : H = 1.000 ; question = humidité ≤ 77.5 ? (gain 1.000) ; oui → Oui (4 O / 0 N) ; non → Non (0 O / 4 N)
Arbre 3 : H = 0.954 ; question = vent fort ? (gain 0.348) ; oui → Non (0 O / 3 N) ; non → Oui (3 O / 2 N)


### Question 3 — Chaque arbre répond, puis on vote
**Réponse.** Arbre 1 : vent fort → **NON** ; arbre 2 : 72 ≤ 77,5 → **OUI** ; arbre 3 : vent fort → **NON**. Vote : 2 NON contre 1 OUI → **on ne joue pas**. Un arbre seul peut se tromper (l’arbre 2 ne voit pas le vent) ; la majorité d’arbres diversifiés est plus fiable.

In [5]:
def repondre(arbre, jour):
    return arbre["feuille_oui"] if arbre["test"](jour) else arbre["feuille_non"]

votes = {nom: repondre(a, demain) for nom, a in foret.items()}
print("Votes :", votes)
decision = majoritaire(list(votes.values()))
print("Décision de la forêt :", decision)
print("Résultat du vote :", {c: list(votes.values()).count(c) for c in set(votes.values())})

Votes : {'Arbre 1': 'Non', 'Arbre 2': 'Oui', 'Arbre 3': 'Non'}
Décision de la forêt : Non
Résultat du vote : {'Oui': 1, 'Non': 2}


## Exemple 2 : régression, ventes de glaces

| T (°C) | 12 | 18 | 26 | 32 |
|---|---|---|---|---|
| Ventes (TND) | 40 | 60 | 200 | 240 |

Échantillons bootstrap (températures) : arbre 1 → 12, 18, 32, 32 ; arbre 2 → 12, 12, 26, 32 ; arbre 3 → 18, 18, 26, 26.

### Question 4 — Apprendre les 3 arbres (plus petite SSE), puis faire la moyenne pour 28 °C
**Réponse.**
- Arbre 1 : ≤ 15 → SSE 21 600 ; ≤ 25 → SSE 200 → seuil **25**, feuilles 50 et 240
- Arbre 2 : ≤ 19 → SSE 800 ; ≤ 29 → SSE 17 067 → seuil **19**, feuilles 40 et 220
- Arbre 3 : un seul seuil possible, ≤ 22 (SSE = 0), feuilles 60 et 200
- 28 °C : 240 ; 220 ; 200 → moyenne $= 660 / 3 =$ **220 TND**

In [6]:
ventes = {12: 40, 18: 60, 26: 200, 32: 240}
echantillons_T = {"Arbre 1": [12, 18, 32, 32], "Arbre 2": [12, 12, 26, 32], "Arbre 3": [18, 18, 26, 26]}

def sse(v):
    m = sum(v) / len(v)
    return sum((x - m) ** 2 for x in v)

def souche_regression(T):
    distincts = sorted(set(T))
    resultats = []
    for a, b in zip(distincts, distincts[1:]):
        s = (a + b) / 2
        g = [ventes[t] for t in T if t <= s]; d = [ventes[t] for t in T if t > s]
        resultats.append((sse(g) + sse(d), s, sum(g) / len(g), sum(d) / len(d)))
    return min(resultats), resultats

predictions = {}
for nom, T in echantillons_T.items():
    (total, seuil, fg, fd), tous = souche_regression(T)
    predictions[nom] = fg if 28 <= seuil else fd
    print(f"{nom} : SSE testées = {[(s, round(t)) for t, s, _, _ in tous]} → seuil {seuil} ; feuilles {fg} / {fd} ; 28 °C → {predictions[nom]}")

moyenne = sum(predictions.values()) / len(predictions)
print(f"Ventes prédites à 28 °C (moyenne des 3 arbres) : ({' + '.join(f'{v:g}' for v in predictions.values())}) / 3 = {moyenne:g} TND")

Arbre 1 : SSE testées = [(15.0, 21600), (25.0, 200)] → seuil 25.0 ; feuilles 50.0 / 240.0 ; 28 °C → 240.0
Arbre 2 : SSE testées = [(19.0, 800), (29.0, 17067)] → seuil 19.0 ; feuilles 40.0 / 220.0 ; 28 °C → 220.0
Arbre 3 : SSE testées = [(22.0, 0)] → seuil 22.0 ; feuilles 60.0 / 200.0 ; 28 °C → 200.0
Ventes prédites à 28 °C (moyenne des 3 arbres) : (240 + 220 + 200) / 3 = 220 TND


---
# Partie B — Les mêmes exemples avec scikit-learn

### Question 5 — Reproduire chaque arbre avec `DecisionTreeClassifier` / `DecisionTreeRegressor`
**Réponse.** Une forêt scikit-learn tire ses échantillons au hasard : on ne peut pas lui imposer ceux de la diapo. Mais chaque arbre d’une forêt est un simple `DecisionTreeClassifier`. Pour reproduire le bootstrap, scikit-learn compte combien de fois chaque exemple a été tiré et utilise ce nombre comme **poids** (`sample_weight`). On fait exactement la même chose : chaque arbre est appris sur les 8 jours, avec pour poids le nombre de tirages, et uniquement sur ses variables tirées.

In [7]:
import numpy as np, pandas as pd
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

noms_jours = list(jours)
X_all = pd.DataFrame({"ciel_soleil": [int(jours[j]["ciel"] == "Soleil") for j in noms_jours],
                      "humidite": [jours[j]["humidite"] for j in noms_jours],
                      "vent_fort": [int(jours[j]["vent"] == "Fort") for j in noms_jours]})
y_all = [jours[j]["joue"] for j in noms_jours]
colonnes = {"ciel": "ciel_soleil", "humidite": "humidite", "vent": "vent_fort"}
demain_df = pd.DataFrame([{"ciel_soleil": 1, "humidite": 72, "vent_fort": 1}])

votes_sk, gains_sk = {}, {}
for nom, (ech, variables) in echantillons.items():
    poids = [Counter(ech)[j] for j in noms_jours]            # nombre de tirages de chaque jour (0 = out-of-bag)
    cols = [colonnes[v] for v in variables]
    arbre = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=0)
    arbre.fit(X_all[cols], y_all, sample_weight=poids)
    votes_sk[nom] = arbre.predict(demain_df[cols])[0]
    t = arbre.tree_
    gain = t.impurity[0] - (t.weighted_n_node_samples[1] * t.impurity[1] + t.weighted_n_node_samples[2] * t.impurity[2]) / t.weighted_n_node_samples[0]
    gains_sk[nom] = gain
    print(f"{nom} : poids = {poids} ; question = {cols[t.feature[0]]} ≤ {t.threshold[0]} ; gain = {gain:.3f} ; demain → {votes_sk[nom]}")
print("Décision (vote) :", Counter(votes_sk.values()).most_common(1)[0][0])

Arbre 1 : poids = [1, 1, 2, 0, 1, 2, 0, 1] ; question = vent_fort ≤ 0.5 ; gain = 0.549 ; demain → Non
Arbre 2 : poids = [1, 2, 0, 1, 1, 0, 2, 1] ; question = humidite ≤ 77.5 ; gain = 1.000 ; demain → Oui
Arbre 3 : poids = [1, 0, 1, 2, 2, 1, 0, 1] ; question = vent_fort ≤ 0.5 ; gain = 0.348 ; demain → Non
Décision (vote) : Non


In [8]:
pred_reg_sk = {}
for nom, T in echantillons_T.items():
    r = DecisionTreeRegressor(max_depth=1).fit(np.array(T).reshape(-1, 1), [ventes[t] for t in T])
    pred_reg_sk[nom] = r.predict([[28]])[0]
    print(f"{nom} : seuil = {r.tree_.threshold[0]} ; 28 °C → {pred_reg_sk[nom]}")
print("Moyenne :", np.mean(list(pred_reg_sk.values())), "TND")

Arbre 1 : seuil = 25.0 ; 28 °C → 240.0
Arbre 2 : seuil = 19.0 ; 28 °C → 220.0
Arbre 3 : seuil = 22.0 ; 28 °C → 200.0
Moyenne : 220.0 TND


### Question 6 — Une vraie forêt scikit-learn (`RandomForestClassifier`, `RandomForestRegressor`)
**Réponse.** Avec 100 arbres tirés au hasard (diapo 45), les échantillons ne sont plus ceux de la diapo, mais la **décision** doit rester la même : on ne joue pas demain. Pour les glaces, on obtient une valeur proche de 220 TND, mais pas exactement 220 : chaque forêt tirée au hasard a vu d’autres échantillons. `oob_score_` estime l’exactitude à partir des jours out-of-bag.

In [9]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

rf = RandomForestClassifier(n_estimators=100, max_features=2, criterion="entropy", oob_score=True,
                            random_state=42, n_jobs=-1).fit(X_all, y_all)
print("Forêt de 100 arbres → demain :", rf.predict(demain_df)[0], "| probabilités :", dict(zip(rf.classes_, rf.predict_proba(demain_df)[0].round(2))))
print(f"Exactitude out-of-bag : {rf.oob_score_:.2f}")

rfr = RandomForestRegressor(n_estimators=100, max_depth=1, random_state=42).fit(np.array(list(ventes)).reshape(-1, 1), list(ventes.values()))
print(f"Forêt de régression (100 arbres, une question) → 28 °C : {rfr.predict([[28]])[0]:.1f} TND")

Forêt de 100 arbres → demain : Non | probabilités : {np.str_('Non'): np.float64(0.72), np.str_('Oui'): np.float64(0.28)}
Exactitude out-of-bag : 0.75
Forêt de régression (100 arbres, une question) → 28 °C : 208.3 TND


---
# Partie C — Comparaison et validation
### Question 7 — Les résultats concordent-ils ?

In [10]:
lignes = [(f"Ex.1 {nom} : gain", round(foret[nom]["gain"], 3), round(gains_sk[nom], 3)) for nom in foret]
lignes += [(f"Ex.1 {nom} → demain", votes[nom], votes_sk[nom]) for nom in foret]
lignes += [("Ex.1 décision (vote)", decision, Counter(votes_sk.values()).most_common(1)[0][0])]
lignes += [(f"Ex.2 {nom} → 28 °C", predictions[nom], pred_reg_sk[nom]) for nom in predictions]
lignes += [("Ex.2 moyenne", moyenne, np.mean(list(pred_reg_sk.values())))]
display(pd.DataFrame(lignes, columns=["Grandeur", "Python pur", "scikit-learn"]))
assert all(abs(foret[n]["gain"] - gains_sk[n]) < 1e-9 for n in foret)
assert votes == votes_sk and predictions == pred_reg_sk
print("✅ Python pur = scikit-learn arbre par arbre : mêmes gains, mêmes réponses, même vote, même moyenne")
print("Forêt scikit-learn de 100 arbres → demain :", rf.predict(demain_df)[0])

,Grandeur,Python pur,scikit-learn
0,Ex.1 Arbre 1 : gain,0.549,0.549
1,Ex.1 Arbre 2 : gain,1.0,1.0
2,Ex.1 Arbre 3 : gain,0.348,0.348
3,Ex.1 Arbre 1 → demain,Non,Non
4,Ex.1 Arbre 2 → demain,Oui,Oui
5,Ex.1 Arbre 3 → demain,Non,Non
6,Ex.1 décision (vote),Non,Non
7,Ex.2 Arbre 1 → 28 °C,240.0,240.0
8,Ex.2 Arbre 2 → 28 °C,220.0,220.0
9,Ex.2 Arbre 3 → 28 °C,200.0,200.0


✅ Python pur = scikit-learn arbre par arbre : mêmes gains, mêmes réponses, même vote, même moyenne
Forêt scikit-learn de 100 arbres → demain : Non


## Conclusion

| Ce qu’on a comparé (Python pur ↔ scikit-learn) | Résultat |
|---|---|
| Ex.1 : gain de la question de chaque arbre (avec les poids du bootstrap) | ✅ identiques |
| Ex.1 : réponse de chaque arbre pour demain et vote final | ✅ identiques |
| Ex.2 : prédiction de chaque arbre à 28 °C et moyenne | ✅ identiques |
| Vraie forêt scikit-learn de 100 arbres (échantillons tirés au hasard) | même décision pour demain |

**À retenir**
- Chaque arbre voit des **exemples différents** (bootstrap) et des **variables différentes** (`max_features`) : les arbres sont **diversifiés**.
- Le vote ou la moyenne réduit l’instabilité d’un arbre seul.
- Dans scikit-learn, le bootstrap est réalisé avec des poids = nombre de tirages de chaque exemple.